# **FEATURE ENGINEERING**

**Pipeline**: Raw CSV → Cleaning (notebook 01) → **Feature Engineering** → EDA → Analytics

In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [14]:
df = pd.read_pickle("Dataset/Cleaned_data/clean_dataset.pkl")

### 1.1 Time Features

**Objective**: Extract time dimensions from `Transaction_Date` to support cyclical analysis (monthly, weekly).

| Feature | Significance |
|---|---|
| `Month` | Month Name (Jan, Feb...) — used for seasonality analysis and joining with the discount table. |
| `Date` | Plain Date (without time) — used for grouping by day.|
| `Week` | ISO Week Period — used for weekly trend analysis. |

In [15]:

df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")
df["Month"]            = df["Transaction_Date"].dt.strftime("%b")
df["Date"]             = df["Transaction_Date"].dt.date
df["Week"]             = df["Transaction_Date"].dt.to_period("W").astype(str)

df["CustomerID"]     = df["CustomerID"].astype(str)
df["Transaction_ID"] = df["Transaction_ID"].astype(str)

### 1.2 Coupon / Discount Flags

**Objective**: Create binary flags to analyze coupon usage behavior and the impact of discounts on revenue.

| Feature |Significance |
|---|---|
| `Is_Discounted` | 1 if the transaction has a discount applied (Discount_pct > 0), 0 otherwise.|
| `Is_Used_Coupon` | 1 if the customer actually redeemed a coupon (Coupon_Status == "Used"), 0 otherwise. |


In [16]:
df["Is_Discounted"]  = (df["Discount_pct"] > 0).astype(int)
df["Is_Used_Coupon"] = (df["Coupon_Status"] == "Used").astype(int)

### 1.3 Invoice — Actual Invoice Value

**Objective**: Calculate the final amount paid by the customer, accounting for all components: quantity, average price, discounts (if a coupon is applied), GST tax, and delivery charges.

**Formulas**:

If a coupon is used:
Invoice = (Quantity x Avg_Price x (1 - Discount_pct) x (1 + GST)) + Delivery_Charges

If no coupon is used:
Invoice = (Quantity x Avg_Price x (1 + GST)) + Delivery_Charges

In [17]:
# Discount_pct da o dang thap phan (0.10), khong chia them /100
df["Invoice"] = np.where(
    df["Coupon_Status"] == "Used",
    (df["Quantity"] * df["Avg_Price"]) * (1 - df["Discount_pct"]) * (1 + df["GST"]) + df["Delivery_Charges"],
    (df["Quantity"] * df["Avg_Price"]) * (1 + df["GST"]) + df["Delivery_Charges"]
)

print("Invoice stats:")
print(df["Invoice"].describe().round(2))

Invoice stats:
count   52924.00
mean      101.98
std       172.37
min         4.60
25%        20.16
50%        45.64
75%       137.40
max      8979.28
Name: Invoice, dtype: float64


### 1.4 Total Revenue — Gross Revenue (for ABC Analysis)

**Objective**: Calculate gross revenue at the transaction level to support ABC classification by product category.

**Formula**: Excluding GST as it represents tax collected on behalf of the government, not actual revenue

In [18]:

df["total_revenue"] = (
    df["Quantity"] * df["Avg_Price"]
    * (1 - df["Discount_pct"])
    + df["Delivery_Charges"]
)

### 1.5 ABC Category — Product Classification based on the Pareto Principle

**Objective**: Apply the Pareto Principle (80/20) to categorize product categories based on their revenue contribution.

| Group | Cumulative Revenue Ratio | Strategic Significance |
|---|---|---|
| **A** | 0 - 80% | Cash cow — prioritize inventory, focus marketing efforts, avoid deep discounts.|
| **B** | 80 - 95% | Potential — monitor closely; aim to push these items into Group A. |
| **C** | 95 - 100% | consider phasing out, liquidating, or bundling.|

In [19]:
def abc_categorizer(df):
    grouped_df = (
        df.groupby("Product_Category", as_index=False)["total_revenue"]
          .sum()
          .rename(columns={"total_revenue": "cat_revenue"})
    )
    grouped_df = grouped_df.sort_values("cat_revenue", ascending=False)

    total_rev = grouped_df["cat_revenue"].sum()
    grouped_df["share"]     = grouped_df["cat_revenue"] / total_rev
    grouped_df["cum_share"] = grouped_df["share"].cumsum()

    def assign_class(cum):
        if cum <= 0.80:
            return "A"
        elif cum <= 0.95:
            return "B"
        else:
            return "C"

    grouped_df["ABC"] = grouped_df["cum_share"].apply(assign_class)
    print(grouped_df[["Product_Category", "cat_revenue", "cum_share", "ABC"]].to_string(index=False))
    return grouped_df.set_index("Product_Category")["ABC"].to_dict()


abc_dict  = abc_categorizer(df)
df["ABC"] = df["Product_Category"].map(abc_dict)
print("\nABC distribution:\n", df["ABC"].value_counts())

    Product_Category  cat_revenue  cum_share ABC
            Nest-USA   2483281.60       0.51   A
             Apparel    728529.50       0.65   A
                Nest    496626.49       0.75   A
              Office    356172.78       0.83   B
           Drinkware    238682.23       0.88   B
                Bags    168002.31       0.91   B
Notebooks & Journals    117342.64       0.93   B
           Lifestyle    112761.45       0.96   C
         Nest-Canada     69578.23       0.97   C
            Headgear     55461.58       0.98   C
          Gift Cards     18521.38       0.99   C
              Google     12226.76       0.99   C
                Waze     11245.97       0.99   C
           Backpacks      9935.61       0.99   C
             Bottles      9854.82       0.99   C
         Accessories      9525.47       1.00   C
                 Fun      7909.32       1.00   C
          Housewares      6508.25       1.00   C
           More Bags      3442.66       1.00   C
             Android

### 1.6 RFM — Recency, Frequency, Monetary

**Objective**: Segment customers based on purchasing behavior. RFM serves as the foundation for churn analysis, CLV (Customer Lifetime Value), and marketing personalization.

| Metric | Definition | Interpretation |
|---|---|---|
| **Recency** | Number of days since the last purchase | Lower = "Hotter" (more active) customer|
| **Frequency** | Number of unique transaction IDs | Higher = More loyal customer |
| **Monetary** | Total Invoice amount paid | Higher = More valuable customer |

**Scoring**:
- `R_Score`: 1 (oldest) to 4 (most recent) — `Inverse relationship` relationship with the raw Recency value.
- `F_Score`, `M_Score`: 1 (lowest) to 4 (highest).
- `RFM_Score` = R + F + M (simple sum, maximum = 12).


In [20]:

today = df["Transaction_Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency   = ("Transaction_Date", lambda x: (today - x.max()).days),
    Frequency = ("Transaction_ID", "nunique"),
    Monetary  = ("Invoice", "sum")
).reset_index()

rfm["R_Score"] = pd.qcut(rfm["Recency"],  q=4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"),  q=4, labels=[1, 2, 3, 4]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

def heuristic_segment(score):
    if score <= 5:
        return "Standard"    # 3-5
    elif score <= 8:
        return "Silver"      # 6-8
    elif score <= 10:
        return "Premium"     # 9-10
    else:
        return "Gold"        # 11-12

rfm["Heuristic_Segment"] = rfm["RFM_Score"].apply(heuristic_segment)

print("RFM stats:")
print(rfm[["Recency","Frequency","Monetary"]].describe().round(2))
print("\nSegment distribution:\n", rfm["Heuristic_Segment"].value_counts())

RFM stats:
       Recency  Frequency  Monetary
count  1468.00    1468.00   1468.00
mean    145.29      18.14   3676.67
std     101.94      24.98   5846.08
min       1.00       1.00      6.99
25%      56.00       5.00    783.97
50%     132.00      11.00   2011.62
75%     221.00      23.00   4495.06
max     365.00     328.00  87200.90

Segment distribution:
 Heuristic_Segment
Silver      494
Standard    407
Premium     315
Gold        252
Name: count, dtype: int64


In [21]:

df = df.merge(
    rfm[[
        "CustomerID", "Recency", "Frequency", "Monetary",
        "R_Score", "F_Score", "M_Score", "RFM_Score", "Heuristic_Segment"
    ]],
    on="CustomerID",
    how="left"
)

### 1.7 Marketing Monthly — Bang tong hop chi phi marketing theo thang

**Objective**: Aggregate monthly revenue and marketing expenditures to analyze Return on Investment (ROI).

| Feature | Significance |
|---|---|
| `Invoice` | Total actual revenue for the month. |
| `Total_Mkt_Spend` | Total marketing expenses (online + offline) for the month. |
| `Pct_Mkt_Spend` | Marketing spend-to-revenue ratio (%) — measures investment efficiency. |
| `Pct_Delivery` | Delivery charges-to-revenue ratio (%) — measures the logistics burden. |

> **Note**: This is a separate aggregated table for analysis and should not be merged back into the main dataframe (df).

In [22]:
marketing_monthly = (
    df.groupby("Month", as_index=False)
      .agg(
          Invoice         = ("Invoice", "sum"),
          Discount_pct    = ("Discount_pct", "mean"),
          GST             = ("GST", "mean"),
          DeliveryCharges = ("Delivery_Charges", "sum"),
          Total_Mkt_Spend = ("Total_Marketing_Spend", "sum")
      )
)

marketing_monthly["Pct_Mkt_Spend"] = (
    marketing_monthly["Total_Mkt_Spend"] * 100 / marketing_monthly["Invoice"]
)
marketing_monthly["Pct_Delivery"] = (
    marketing_monthly["DeliveryCharges"] * 100 / marketing_monthly["Invoice"]
)

print(marketing_monthly.round(2))

   Month   Invoice  Discount_pct  GST  DeliveryCharges  Total_Mkt_Spend  \
0    Apr 477498.59          0.03 0.14         41481.74      21655922.13   
1    Aug 475796.88          0.07 0.15         61099.57      28385733.77   
2    Dec 556112.29          0.10 0.12         37881.99      28964402.01   
3    Feb 375162.05          0.07 0.14         49216.60      15841536.05   
4    Jan 494090.55          0.03 0.13         59242.32      20052775.17   
5    Jul 451878.41          0.03 0.14         48723.93      20618934.41   
6    Jun 361000.17          0.10 0.14         37513.58      18625403.73   
7    Mar 415157.79          0.10 0.14         60799.94      17453780.31   
8    May 365596.03          0.06 0.14         41396.17      17525521.02   
9    Nov 547788.13          0.07 0.12         32311.93      21096299.69   
10   Oct 480767.37          0.03 0.13         45961.88      20536272.39   
11   Sep 396510.49          0.10 0.14         41005.42      19257626.34   

    Pct_Mkt_Spend  Pct_D

### 1.8 K-Means Segmentation — Customer Clustering (Optional)

**Objective**: Utilize the K-Means algorithm to cluster customers based on RFM metrics, supplementing the heuristic segments.

| Feature | Significance |
|---|---|
| `KMeans_Label` | NCluster Label (0, 1, 2, 3) — requires further interpretation after examining the centroids.|

**Why is StandardScaler necessary?** Recency, Frequency, and Monetary have very different units and scales. Without scaling, K-Means will be dominated by the Monetary value and effectively ignore Frequency.

> **Note**: `k=4` is the default choice. It is recommended to verify this using the Elbow Method or Silhouette Score first.

In [23]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

kmeans_features = rfm[["Recency", "Frequency", "Monetary"]].copy()
scaler        = StandardScaler()
kmeans_scaled = scaler.fit_transform(kmeans_features)

k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
rfm["KMeans_Label"] = kmeans.fit_predict(kmeans_scaled)

df = df.merge(rfm[["CustomerID", "KMeans_Label"]], on="CustomerID", how="left")

print("KMeans cluster profile:")
print(rfm.groupby("KMeans_Label")[["Recency","Frequency","Monetary"]].mean().round(2))

KMeans cluster profile:
              Recency  Frequency  Monetary
KMeans_Label                              
0               80.26      55.69  11932.70
1              255.17      10.29   1969.64
2               79.25      12.79   2433.38
3               81.83     270.67  65739.45


---
## **2. Export master_df.csv**

Export **only once** at this stage with all calculated columns included.
For EDA and subsequent notebooks, **simple use `pd.read_csv("Dataset/Cleaned_data/master_df.csv")`** 

|Column Group | Columns |
|---|---|
| Identity | `CustomerID`, `Transaction_ID`, `Transaction_Date` |
| Time | `Month`, `Month_Num`, `Year`, `YearMonth`, `Date`, `Week`, `DayName`, `Season` |
| Product | `Product_SKU`, `Product_Description`, `Product_Category`, `ABC` |
| Financial | `Quantity`, `Avg_Price`, `Delivery_Charges`, `Revenue`, `total_revenue`, `Invoice`, `Discount_pct`, `Discount_Amount`, `GST` |
| Customer | `Gender`, `Location`, `Tenure_Months` |
| Coupon | `Coupon_Status`, `Is_Discounted`, `Is_Used_Coupon`, `Coupon_Used` |
| Marketing | `Offline_Spend`, `Online_Spend`, `Total_Marketing_Spend` |
| RFM | `Recency`, `Frequency`, `Monetary`, `R_Score`, `F_Score`, `M_Score`, `RFM_Score`, `Heuristic_Segment`, `KMeans_Label` |


In [26]:
# ── 2. Export master_df.csv ──────────────────────────────────────────────────
import os
os.makedirs("Dataset/Cleaned_data", exist_ok=True)

# Danh sách cột muốn export (theo thứ tự logic)
export_cols = [
    # Identity
    "CustomerID", "Transaction_ID", "Transaction_Date",
    # Time — đầy đủ để EDA dùng trực tiếp
    "Month", "Month_Num", "Year", "YearMonth", "Date", "Week", "DayName", "Season",
    # Product
    "Product_SKU", "Product_Description", "Product_Category", "ABC",
    # Financial
    "Quantity", "Avg_Price", "Delivery_Charges",
    "Revenue",          # Qty × Price (pre-tax, pre-discount) — raw
    "total_revenue",    # Qty × Price × (1-discount) × (1+GST) — basis ABC
    "Invoice",          # total_revenue + Delivery — tiền khách thực trả
    "Discount_pct", "Discount_Amount", "GST",
    # Customer
    "Gender", "Location", "Tenure_Months",
    # Coupon
    "Coupon_Status", "Is_Discounted", "Is_Used_Coupon", "Coupon_Used",
    # Marketing
    "Offline_Spend", "Online_Spend", "Total_Marketing_Spend",
    # RFM
    "Recency", "Frequency", "Monetary",
    "R_Score", "F_Score", "M_Score", "RFM_Score",
    "Heuristic_Segment", "KMeans_Label",
]

# Chỉ export cột thực sự tồn tại (tránh KeyError nếu KMeans bị skip)
actual_cols = [c for c in export_cols if c in df.columns]
missing_export = [c for c in export_cols if c not in df.columns]
if missing_export:
    print(f"⚠️  Cột chưa có (sẽ không export): {missing_export}")

master_df = df[actual_cols].copy()
master_df.to_pickle("Dataset/Cleaned_data/master_df.pkl")

print(f"✅ Exported master_df.pkl: {master_df.shape[0]:,} rows × {master_df.shape[1]} cols")
print(f"   Path: Dataset/Cleaned_data/master_df.pkl")
print()
print("── Cột đã export ──")
for g, cols in [
    ("Identity  ", ["CustomerID","Transaction_ID","Transaction_Date"]),
    ("Time      ", ["Month","Month_Num","Year","YearMonth","Date","Week","DayName","Season"]),
    ("Product   ", ["Product_SKU","Product_Description","Product_Category","ABC"]),
    ("Financial ", ["Revenue","total_revenue","Invoice","Discount_pct","Discount_Amount","GST","Delivery_Charges"]),
    ("Customer  ", ["Gender","Location","Tenure_Months"]),
    ("Coupon    ", ["Coupon_Status","Is_Discounted","Is_Used_Coupon","Coupon_Used"]),
    ("Marketing ", ["Offline_Spend","Online_Spend","Total_Marketing_Spend"]),
    ("RFM       ", ["Recency","Frequency","Monetary","R_Score","F_Score","M_Score","RFM_Score","Heuristic_Segment","KMeans_Label"]),
]:
    present = [c for c in cols if c in actual_cols]
    print(f"  {g}: {present}")


⚠️  Cột chưa có (sẽ không export): ['Month_Num', 'Year', 'YearMonth', 'DayName', 'Season', 'Discount_Amount', 'Coupon_Used']
✅ Exported master_df.pkl: 52,924 rows × 36 cols
   Path: Dataset/Cleaned_data/master_df.pkl

── Cột đã export ──
  Identity  : ['CustomerID', 'Transaction_ID', 'Transaction_Date']
  Time      : ['Month', 'Date', 'Week']
  Product   : ['Product_SKU', 'Product_Description', 'Product_Category', 'ABC']
  Financial : ['Revenue', 'total_revenue', 'Invoice', 'Discount_pct', 'GST', 'Delivery_Charges']
  Customer  : ['Gender', 'Location', 'Tenure_Months']
  Coupon    : ['Coupon_Status', 'Is_Discounted', 'Is_Used_Coupon']
  Marketing : ['Offline_Spend', 'Online_Spend', 'Total_Marketing_Spend']
  RFM       : ['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Heuristic_Segment', 'KMeans_Label']
